In [1]:
import os
import pickle
import re
import sys

import pandas as pd
from tqdm import tqdm

from connectDriveCloud import authenticate_google_drive, load_pickle_content, get_files
from distances import fidelityCalc, traceDist, getHellinger, compareChisquare, jensenShannonDivergence

In [2]:
origin_id = "1MTTleRgnFJ2UnYmbpzZoh2ndmWBJ3YJk"
mutants_id = "1TuXmlQAARKVeOm4nTWSJBmI500Tns4aZ" # equivalent_mutant dir

service = authenticate_google_drive()

In [3]:
def get_files_id_dict(service, folder_id):
    items = get_files(service, folder_id)
    return {item['name']: item['id'] for item in items} if items else {}

In [4]:
def load_and_merge_files(service, folder_id):
    items = get_files(service, folder_id)
    merged_data = []

    for item in items:
        filename = item['name']
        file_id = item['id']

        try:
            data = load_pickle_content(service, file_id)
            merged_data.extend(data)
        except pickle.UnpicklingError:
            print(f'Error unpickling file: {filename}')
        except Exception as e:
            print(f'Error processing file {filename}: {str(e)}')

    return merged_data

In [5]:
def get_df(oracle_data, mutants_data):
    column_names = ['Name', 'Input', 'Ideal_chisquare', 'Ideal_hellinger',
                    'Ideal_jensenshannon', 'Ideal_trace', 'Ideal_fidelity',
                     'Ideal_expectation', 'Killed_IC', 'Killed_IH', 'Killed_IJ',
                    'Killed_IT', 'Killed_IF', 'Killed_IE']

    results = []

    # Convert the oracle_data list of dictionaries into a lookup dictionary for faster access
    oracle_lookup = {item['Input']: item for item in oracle_data}

    for mutant in mutants_data:
        input_value = mutant['Input']
        if input_value in oracle_lookup:
            oracle_entry = oracle_lookup[input_value]

            # Perform calculations
            ideal_chisquare = compareChisquare(oracle_entry['Ideal_output_distribution'],
                                               mutant['Ideal_output_distribution'])

            ideal_hellinger = getHellinger(oracle_entry['Ideal_output_distribution'],
                                           mutant['Ideal_output_distribution'])

            ideal_jensenshannon = jensenShannonDivergence(oracle_entry['Ideal_output_distribution'],
                                               mutant['Ideal_output_distribution'])

            ideal_fidelity = fidelityCalc(oracle_entry['Ideal_density_matrix'], mutant['Ideal_density_matrix'])

            ideal_trace = traceDist(oracle_entry['Ideal_density_matrix'], mutant['Ideal_density_matrix'])

            ideal_expectation = abs(oracle_entry['Ideal_expectation_value']-mutant['Ideal_expectation_value'])


            # Create a dictionary for the result row
            new_line = {
                'Name': mutant['Name'].split('/')[-1],
                'Input': mutant['Input'],
                'Ideal_chisquare': ideal_chisquare,
                'Ideal_hellinger': ideal_hellinger,
                'Ideal_jensenshannon': ideal_jensenshannon,
                'Ideal_trace': ideal_trace,
                'Ideal_fidelity': ideal_fidelity,
                'Ideal_expectation': ideal_expectation
            }

            results.append(new_line)

    # Convert the results list of dictionaries to a DataFrame
    results_df = pd.DataFrame(results, columns=column_names)

    return results_df

In [ ]:
origin_files = get_files(service, origin_id)
dic_mutant_folders = get_files_id_dict(service, mutants_id)

column_names = ['Name', 'Input', 'Ideal_chisquare', 'Ideal_hellinger', 'Ideal_jensenshannon', 'Ideal_trace', 'Ideal_fidelity',
                'Ideal_expectation', 'Killed_IC', 'Killed_IH', 'Killed_IJ', 'Killed_IT', 'Killed_IF', 'Killed_IE']
results_df = pd.DataFrame(columns=column_names)

for item in tqdm(origin_files, desc="Checking results..."):
    filename = item['name']
    file_id = item['id']
    if filename.endswith('.pkl'):
            try:
                pattern = r"indep_qiskit_|_output|.pkl"
                circuit_name = re.sub(pattern, "", filename)
                qubits = int(circuit_name.split('_')[1])
                if qubits <= 8:
                    print(circuit_name)
                    oracle_pkl = load_pickle_content(service, file_id)
                    if isinstance(oracle_pkl, list):
                        mutant_folder_id = dic_mutant_folders.get(f'selected_equivalent_mutants_{circuit_name}')
                        if mutant_folder_id:
                            mutants_pkl = load_and_merge_files(service, mutant_folder_id)
                            df_temp = get_df(oracle_pkl, mutants_pkl)  
                            results_df = pd.concat([results_df, df_temp], ignore_index=True)
                        else:
                            print(f"No mutant folder found for {circuit_name}")
                    else:
                        print(f"Pickle file should contain a List instead of a {type(oracle_pkl)}.")
                        sys.exit(1)
            except pickle.UnpicklingError:
                print(f'Error unpickling file: {filename}')
            except Exception as e:
                print(f'Error processing file {filename}: {str(e)}')
            
print(results_df.head())

Checking results...:   0%|                                                               | 0/41 [00:00<?, ?it/s]

qftentangled_8


/var/folders/p8/8yxfrvfn7_xdjwr5tlw70gqw0000gq/T/ipykernel_27300/381045504.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, df_temp], ignore_index=True)
Checking results...:   2%|█▏                                                | 1/41 [30:51<20:34:25, 1851.65s/it]

qft_8


In [ ]:
print(results_df.head())

In [ ]:
os.makedirs(f'results_custom_brisbane/results_equiv', exist_ok=True)
results_df.to_csv(f'results_custom_brisbane/results_equiv/results.csv')

In [8]:
trace_error_rounded = 1E-13
fidelity_error_rounded = 1E-14

In [9]:
print(len(results_df))

unique_count = results_df['Name'].nunique()
print(unique_count)

10648
461


In [ ]:
# Confirmed mutants
unique_names_non_zero = results_df[results_df['Ideal_expectation'] != 0]
count_non_zero = len(unique_names_non_zero['Name'].unique())
print(unique_names_non_zero['Name'].unique())
print("Confirmed mutants:", count_non_zero)

# Potential true equivalents
df_mutant_candidates = results_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] == 0).all())['Name'].unique()
count_zero = len(df_mutant_candidates)
print("Potential true equivalents:", count_zero)

df_mutant_candidates = results_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] == 0).all())

df_filtered = df_mutant_candidates[(1 - df_mutant_candidates['Ideal_fidelity'] > fidelity_error_rounded)]
print(df_filtered['Name'].unique())
print(df_filtered['Name'].nunique())

df_filtered = df_mutant_candidates[(df_mutant_candidates['Ideal_trace'] > trace_error_rounded)]
print(df_filtered['Name'].unique())
print(df_filtered['Name'].nunique())

In [ ]:
# Count of unique names where all rows Ideal_expectation == 0
unique_names_zero = results_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] == 0).all())['Name'].unique()
count_zero = len(unique_names_zero)

# Count of unique names where all rows Ideal_expectation != 0
unique_names_non_zero = results_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] != 0).all())['Name'].unique()
count_non_zero = len(unique_names_non_zero)

# Count of unique names where there are both 0 and non-0 rows
# This is done by checking names that have at least one row with 0 and one row with non-zero expectation
unique_names_both = results_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] == 0).any() and (x['Ideal_expectation'] != 0).any())['Name'].unique()
count_both = len(unique_names_both)

# Print the counts
print("Count of unique names where all rows Ideal_expectation == 0:", count_zero)
print("Count of unique names where all rows Ideal_expectation != 0:", count_non_zero)
print("Count of unique names where there are both 0 and non-0 rows:", count_both)
count_all = count_zero + count_non_zero + count_both
print("Sum:", count_all)

total_unique_count = results_df['Name'].nunique()
print("Total unique names in results_df:", total_unique_count)


In [ ]:
# Shows that fidelity > trace and  expectation != fidelity
df_filtered = results_df[(results_df['Ideal_expectation'] == 0) & (1 - results_df['Ideal_fidelity'] > fidelity_error_rounded)]
print(len(df_filtered))
print(df_filtered['Name'].nunique())

df_filtered = df_filtered[(df_filtered['Ideal_trace'] > trace_error_rounded)]
print(len(df_filtered))


df_filtered = results_df[(results_df['Ideal_expectation'] == 0) & (results_df['Ideal_trace'] > trace_error_rounded)]
print(len(df_filtered))


In [ ]:
# Shows that expectation > fidelity and trace
df_filtered = results_df[(results_df['Ideal_expectation'] != 0)]
print(len(df_filtered))

df_filtered = df_filtered[(1 - df_filtered['Ideal_fidelity'] > fidelity_error_rounded)]
print(len(df_filtered))

df_filtered = df_filtered[(df_filtered['Ideal_trace'] > trace_error_rounded)]
print(len(df_filtered))

In [ ]:
def calculate_killed_flags(ideal, tolerance_values_ideal):
    """
    Determines the killed flags based on ideal and noisy values and tolerance values.
    """
    killed_flags = {}
    killed_flags['Killed_IF'] = ideal['fidelity'] < tolerance_values_ideal['fidelity']
    killed_flags['Killed_IT'] = ideal['trace'] > tolerance_values_ideal['trace']
    killed_flags['Killed_IH'] = ideal['hellinger'] > tolerance_values_ideal['hellinger']
    killed_flags['Killed_IC'] = ideal['chisquare'] < tolerance_values_ideal['chisquare']
    killed_flags['Killed_IJ'] = ideal['jensenshannon'] > tolerance_values_ideal['jensenshannon']
    killed_flags['Killed_IE'] = ideal['expectation'] > tolerance_values_ideal['expectation']
    return pd.DataFrame(killed_flags)

In [ ]:
# Define tolerance values
tolerance_values_ideal = {
    'fidelity': 1, # - 0.0000001,
    'trace': 0, # .0000001,
    'hellinger': 0.1,
    'jensenshannon': 0.1,
    'chisquare': 0.01, #.0000001,
    'expectation': 0 #.0000001
}


In [ ]:
# Determine killed flags
killed_flags_df = calculate_killed_flags(
    ideal={'fidelity': results_df['Ideal_fidelity'], 'trace': results_df['Ideal_trace'], 'hellinger': results_df['Ideal_hellinger'],
            'chisquare': results_df['Ideal_chisquare'], 'jensenshannon': results_df['Ideal_jensenshannon'], 'expectation': results_df['Ideal_expectation']},
    tolerance_values_ideal=tolerance_values_ideal
)

# Fill the existing NaN columns in results_df with calculated flags
for flag in killed_flags_df.columns:
    results_df[flag] = killed_flags_df[flag]

In [ ]:
print(results_df.head())